<!-- type: how-to -->
# ForecastPrepContract — JSON Serialisation Roundtrip

This recipe demonstrates the **framework-agnostic hand-off boundary** of the
`ForecastPrepContract`:

1. Build a contract from a triage run.
2. Serialise to JSON and write to disk.
3. Reload and re-validate the Pydantic model.
4. Consume the raw JSON with **no** `forecastability` import — proving the
   contract is a plain data object that any downstream framework can read.


In [ ]:
import json
from pathlib import Path

import numpy as np

from forecastability import (
    ForecastPrepContract,
    TriageRequest,
    build_forecast_prep_contract,
    generate_ar1,
    run_forecastability_fingerprint,
    run_triage,
)
from forecastability.triage import AnalysisGoal

SEED = 42
OUTPUT_ROOT = Path("outputs/notebooks/recipes/contract_roundtrip")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## 1. Build the contract

In [ ]:
series = generate_ar1(n_samples=300, phi=0.85, random_state=SEED)

triage_result = run_triage(
    TriageRequest(
        series=series,
        goal=AnalysisGoal.univariate,
        max_lag=20,
        n_surrogates=99,
        random_state=SEED,
    )
)

fp_bundle = run_forecastability_fingerprint(
    series,
    target_name="synthetic_ar1",
    max_lag=20,
    n_surrogates=99,
    random_state=SEED,
)

contract = build_forecast_prep_contract(
    triage_result,
    horizon=12,
    target_frequency="MS",
    fingerprint_bundle=fp_bundle,
    add_calendar_features=False,
)

print(f"Contract built — blocked={contract.blocked}, horizon={contract.horizon}")
print(f"Recommended families: {contract.recommended_families}")
print(f"Recommended target lags: {contract.recommended_target_lags}")


## 2. Serialise to JSON and write to disk

In [ ]:
json_path = OUTPUT_ROOT / "contract.json"
json_str = contract.model_dump_json(indent=2)
json_path.write_text(json_str, encoding="utf-8")

print(f"Written to: {json_path}")
print(f"File size: {json_path.stat().st_size} bytes")
print()
print("First 400 characters of the JSON:")
print(json_str[:400])


## 3. Reload and re-validate with Pydantic

In [ ]:
# Reload from disk using Pydantic model_validate_json
reloaded_json = json_path.read_text(encoding="utf-8")
reloaded_contract = ForecastPrepContract.model_validate_json(reloaded_json)

# Verify round-trip fidelity
assert reloaded_contract.recommended_target_lags == contract.recommended_target_lags, \
    "Lag mismatch after roundtrip"
assert reloaded_contract.recommended_families == contract.recommended_families, \
    "Family mismatch after roundtrip"
assert reloaded_contract.horizon == contract.horizon, \
    "Horizon mismatch after roundtrip"
assert reloaded_contract.blocked == contract.blocked, \
    "Blocked flag mismatch after roundtrip"

print("Round-trip validation passed.")
print(f"  recommended_target_lags: {reloaded_contract.recommended_target_lags}")
print(f"  recommended_families:    {reloaded_contract.recommended_families}")
print(f"  horizon:                 {reloaded_contract.horizon}")
print(f"  forecastability_class:   {reloaded_contract.forecastability_class}")


## 4. Consume the raw JSON — no `forecastability` import

This final section demonstrates the **framework-agnostic hand-off boundary**:
the contract JSON is a plain dictionary. A downstream framework (or a separate
service) can consume it with only the standard library.


In [ ]:
# This cell imports only the standard library — no forecastability dependency.
import json as _json
from pathlib import Path as _Path

_raw = _json.loads(
    _Path("outputs/notebooks/recipes/contract_roundtrip/contract.json").read_text()
)

print("Framework-agnostic consumption (standard library only):")
print(f"  contract_version:        {_raw['contract_version']}")
print(f"  blocked:                 {_raw['blocked']}")
print(f"  horizon:                 {_raw['horizon']}")
print(f"  recommended_target_lags: {_raw['recommended_target_lags']}")
print(f"  recommended_families:    {_raw['recommended_families']}")
print(f"  target_frequency:        {_raw['target_frequency']!r}")
print()
print("A downstream MLForecast setup derived purely from the JSON:")
_lags = _raw["recommended_target_lags"] or [1]
print(f"  mlforecast lags = {_lags}")
print(f"  mlforecast freq = {_raw['target_frequency']!r}")
print()
print("No forecastability imports were used in this cell.")
